In [1]:
import numpy as np
import pandas as pd



In [2]:
dataset_path = "../datasets/diginetica"
griffin_dataset_path = "../griffin_datasets/diginetica-downsample"

In [3]:
ctr_train = pd.read_parquet(f"{dataset_path}/ctr/ctr-100k_train.pqt")
ctr_validation = pd.read_parquet(f"{dataset_path}/ctr/ctr-100k_validation.pqt")
ctr_test = pd.read_parquet(f"{dataset_path}/ctr/ctr-100k_test.pqt")
# Update the clicked column to be a boolean
ctr_train["clicked"] = ctr_train["clicked"].astype(bool)
ctr_validation["clicked"] = ctr_validation["clicked"].astype(bool)
ctr_test["clicked"] = ctr_test["clicked"].astype(bool)
ctr_train['click_id'] = ctr_train.index
ctr_validation['click_id'] = ctr_validation.index
ctr_test['click_id'] = ctr_test.index


In [4]:
# construct a click_querytest dataframe combines ctr_train, ctr_validation, ctr_test
click_querytest = pd.concat([ctr_train, ctr_validation, ctr_test])
click_querytest

,queryId,itemId,timestamp,clicked,click_id
80440533,861044,468709,2016-08-05 06:58:49.079,False,80440533
80440420,861044,36016,2016-08-05 06:58:49.079,False,80440420
80440419,861044,36015,2016-08-05 06:58:49.079,False,80440419
80440418,861044,36014,2016-08-05 06:58:49.079,False,80440418
80440417,861044,36012,2016-08-05 06:58:49.079,False,80440417
...,...,...,...,...,...
19192252,240200,78132,2016-10-26 18:50:30.578,False,19192252
19192253,240200,87938,2016-10-26 18:50:30.578,False,19192253
19192254,240200,122178,2016-10-26 18:50:30.578,False,19192254
19192248,240200,66576,2016-10-26 18:50:30.578,False,19192248


In [5]:
# Downsample the QueryResult dataframe
query_results = pd.read_parquet(f"{dataset_path}/data/query_results.pqt")

In [6]:
# Downsample the QueryResult dataframe
query_results = query_results.sample(frac=0.1, random_state=42)


In [7]:
query_results

,queryId,itemId,timestamp
109572,101333,9231,2016-05-14 03:55:36.557
604415,698225,19652,2016-03-15 19:45:58.098
441658,527341,16587,2016-02-06 02:55:24.838
469871,289877,128990,2016-05-29 22:42:19.089
851896,821678,174478,2016-04-11 17:03:11.515
...,...,...,...
155066,821816,36333,2016-05-30 19:07:20.504
146067,79839,13497,2016-03-25 01:54:27.152
490488,552221,188130,2016-02-04 03:53:10.591
466939,709132,89753,2016-04-24 09:26:28.984


In [10]:
# Save the parquet files under griffin_dataset
import shutil
import os
shutil.rmtree(griffin_dataset_path, ignore_errors=True)
shutil.copytree(dataset_path, griffin_dataset_path)

'../griffin_datasets/diginetica-downsample'

In [11]:
# remove the ctr folder
shutil.rmtree(f"{griffin_dataset_path}/ctr", ignore_errors=False)
os.mkdir(f"{griffin_dataset_path}/ctr")
# remove the query_results file
# if is a directory, remove it; if is a file, remove it
if os.path.isdir(f"{griffin_dataset_path}/data/query_results.pqt"):
    shutil.rmtree(f"{griffin_dataset_path}/data/query_results.pqt", ignore_errors=False)
else:
    os.remove(f"{griffin_dataset_path}/data/query_results.pqt")
# Save the parquet files under griffin dataset
query_results.to_parquet(f"{griffin_dataset_path}/data/query_results.pqt")
# Save to the griffin dataset
ctr_train.to_parquet(f"{griffin_dataset_path}/ctr/ctr-100k_train.pqt")
ctr_validation.to_parquet(f"{griffin_dataset_path}/ctr/ctr-100k_validation.pqt")
ctr_test.to_parquet(f"{griffin_dataset_path}/ctr/ctr-100k_test.pqt")

click_querytest.to_parquet(f"{griffin_dataset_path}/data/click_querytest.pqt")


dataset_name: diginetica-downsample
tables:
  - name: Product
    source: data/products.pqt
    format: parquet
    columns:
      - name: itemId
        dtype: primary_key
      - name: categoryId
        dtype: category
      - name: pricelog2
        dtype: float
  - name: Click
    source: data/clicks.pqt
    format: parquet
    columns:
      - name: queryId
        dtype: foreign_key
        link_to: Query.queryId
      - name: itemId
        dtype: foreign_key
        link_to: Product.itemId
      - name: timestamp
        dtype: datetime
    time_column: timestamp
  - name: ClickQueryTest
    source: data/click_querytest.pqt
    format: parquet
    columns:
      - name: click_id
        dtype: primary_key
      - name: queryId
        dtype: foreign_key
        link_to: Query.queryId
      - name: itemId
        dtype: foreign_key
        link_to: Product.itemId
      - name: clicked
        dtype: category
      - name: timestamp
        dtype: datetime
    time_column: timestamp
  - name: QueryResult
    source: data/query_results.pqt
    format: parquet
    columns:
      - name: queryId
        dtype: foreign_key
        link_to: Query.queryId
      - name: itemId
        dtype: foreign_key
        link_to: Product.itemId
      - name: timestamp
        dtype: datetime
    time_column: timestamp
  - name: View
    source: data/item_views.pqt
    format: parquet
    columns:
      - name: sessionId
        dtype: foreign_key
        link_to: Session.id
      - name: userId
        dtype: foreign_key
        link_to: User.id
      - name: itemId
        dtype: foreign_key
        link_to: Product.itemId
      - name: timestamp
        dtype: datetime
    time_column: timestamp
  - name: Purchase
    source: data/purchases.pqt
    format: parquet
    columns:
      - name: sessionId
        dtype: foreign_key
        link_to: Session.id
      - name: userId
        dtype: foreign_key
        link_to: User.id
      - name: itemId
        dtype: foreign_key
        link_to: Product.itemId
      - name: ordernumber
        dtype: foreign_key
        link_to: Orders.id
      - name: timestamp
        dtype: datetime
    time_column: timestamp
  - name: Query
    source: data/queries.pqt
    format: parquet
    columns:
      - name: queryId
        dtype: primary_key
      - name: sessionId
        dtype: foreign_key
        link_to: Session.id
      - name: userId
        dtype: foreign_key
        link_to: User.id
      - name: duration
        dtype: float
      - name: categoryId
        dtype: category
      - name: timestamp
        dtype: datetime
    time_column: timestamp
  - name: ProductNameToken
    source: data/product_name_tokens.pqt
    format: parquet
    columns:
      - name: itemId
        dtype: foreign_key
        link_to: Product.itemId
      - name: token
        dtype: foreign_key
        link_to: Token.id
  - name: QuerySearchstringToken
    source: data/query_searchstring_tokens.pqt
    format: parquet
    columns:
      - name: queryId
        dtype: foreign_key
        link_to: Query.queryId
      - name: token
        dtype: foreign_key
        link_to: Token.id
tasks:
  - name: ctr
    source: ctr/ctr-100k_{split}.pqt
    format: parquet
    columns:
      - name: click_id
        dtype: primary_key
      - name: itemId
        dtype: foreign_key
        link_to: Product.itemId
      - name: queryId
        dtype: foreign_key
        link_to: Query.queryId
      - name: timestamp
        dtype: datetime
      - name: clicked
        dtype: category
    time_column: timestamp
    evaluation_metric: auroc
    target_column: clicked
    target_table: ClickQueryTest
    task_type: classification
  - name: purchase
    source: purchase/purchase_{split}.pqt
    format: parquet
    columns:
      - name: itemId
        dtype: foreign_key
        link_to: Product.itemId
      - name: sessionId
        dtype: foreign_key
        link_to: Session.id
      - name: timestamp
        dtype: datetime
    time_column: timestamp
    evaluation_metric: mrr
    target_column: itemId
    target_table: Purchase
    task_type: retrieval
